# Dataset Comparison

Comprehensive comparison of all raw datasets across size, feature coverage, data quality, and PricePilot relevance.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Find project root dynamically - works on any machine after cloning
notebook_dir = Path.cwd()
if 'eda' not in str(notebook_dir):
    notebook_dir = Path.cwd().parents[0]
project_root = notebook_dir.parents[0] if (notebook_dir / 'notebooks').exists() else notebook_dir.parents[1]
report_dir = project_root / 'eda' / 'reports'

# Load inventory
inventory_path = report_dir / 'dataset_inventory_summary.csv'
inventory = pd.read_csv(inventory_path)
print("Dataset Inventory")
print(f"Total datasets: {len(inventory)}")
display(inventory)


In [ ]:

# Dataset dimensions comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Row counts
inv_sorted = inventory.sort_values('rows', ascending=True)
axes[0].barh(inv_sorted['file_name'], inv_sorted['rows'])
axes[0].set_xlabel('Number of Rows')
axes[0].set_title('Dataset Row Counts (sorted)')
axes[0].ticklabel_format(style='plain', axis='x')

# Column counts
inv_sorted_cols = inventory.sort_values('columns', ascending=True)
axes[1].barh(inv_sorted_cols['file_name'], inv_sorted_cols['columns'])
axes[1].set_xlabel('Number of Columns')
axes[1].set_title('Dataset Column Counts (sorted)')

plt.tight_layout()
plt.show()


In [ ]:

# Feature coverage analysis
features_df = pd.DataFrame({
    'Dataset': inventory['file_name'],
    'Has Price': inventory['price_columns'].notna() & (inventory['price_columns'] != ''),
    'Has Sales/Qty': inventory['sales_columns'].notna() & (inventory['sales_columns'] != ''),
    'Has Date': inventory['date_columns'].notna() & (inventory['date_columns'] != ''),
    'Has Identifiers': inventory['identifier_columns'].notna() & (inventory['identifier_columns'] != ''),
})

print("PricePilot Feature Coverage by Dataset")
display(features_df)

# Coverage percentage
coverage_cols = ['Has Price', 'Has Sales/Qty', 'Has Date', 'Has Identifiers']
features_df['Feature Coverage %'] = (features_df[coverage_cols].sum(axis=1) / len(coverage_cols) * 100).round(1)
display(features_df[['Dataset', 'Feature Coverage %']])


In [ ]:

# Dataset categorization
retail_keywords = ['sales', 'markdown', 'discount', 'price', 'actual', 'demand', 'promotions']
ecommerce_keywords = ['amazon', 'ecommerce', 'online']
reference_keywords = ['catalog', 'stores', 'product']

categories = []
for idx, row in inventory.iterrows():
    name = row['file_name'].lower()
    if any(k in name for k in retail_keywords):
        cat = 'Retail Analytics'
    elif any(k in name for k in ecommerce_keywords):
        cat = 'Ecommerce Analytics'
    else:
        cat = 'Reference/Metadata'
    categories.append(cat)

inventory['Category'] = categories

print("Datasets by Category")
category_summary = inventory.groupby('Category')[['rows', 'columns']].agg({'rows': 'sum', 'columns': 'mean'})
display(category_summary)

print("\nDetailed Classification")
for cat in inventory['Category'].unique():
    print(f"\n{cat}:")
    for f in inventory[inventory['Category'] == cat]['file_name']:
        print(f"  - {f}")


In [ ]:

# Data quality indicators
print("Data Quality Summary")
print("\nNote: Data quality is assessed during individual dataset EDA.")
print("Review the individual dataset reports for:")
print("- Missing value percentages")
print("- Duplicate row detection")
print("- Date range and temporal coverage")
print("- Outlier analysis")
print("\nReports location: eda/reports/")


In [ ]:

# Dataset integration readiness analysis
print("Dataset Integration Readiness Assessment")
print("\nBefore merging datasets, analyze:")
print("1. Common key fields (product_id, store_id, date, etc.)")
print("2. Data granularity alignment (transaction vs. summary level)")
print("3. Temporal overlap between datasets")
print("4. Data type consistency")
print("5. Potential duplicate keys")
print("\nNote: Individual dataset reports include key field analysis.")
print("Cross-dataset integration is documented in individual report recommendations.")


In [ ]:

# Recommendations for PricePilot
print("RECOMMENDATIONS FOR PRICEPILOT PROJECT")
print("=" * 60)
print("\nPRIMARY DATASETS (Most relevant for pricing):")
for idx, row in inventory[inventory['potential_role'].str.contains('Retail')].iterrows():
    print(f"  ✓ {row['file_name']}")
    print(f"    Rows: {row['rows']:,} | Columns: {row['columns']}")
    print(f"    Features: {row['price_columns'] if pd.notna(row['price_columns']) else 'None'}")

print("\nSUPPORTING DATASETS (Context/metadata):")
for idx, row in inventory[~inventory['potential_role'].str.contains('Retail|Ecommerce')].iterrows():
    print(f"  • {row['file_name']}")
    print(f"    Rows: {row['rows']:,} | Columns: {row['columns']}")

print("\nECOMMERCE DATASETS (Alternative/parallel analysis):")
for idx, row in inventory[inventory['potential_role'].str.contains('Ecommerce')].iterrows():
    print(f"  ◦ {row['file_name']}")
    print(f"    Rows: {row['rows']:,} | Columns: {row['columns']}")

print("\nNEXT STEPS:")
print("  1. Review individual dataset EDA reports in eda/reports/")
print("  2. Validate temporal consistency across datasets")
print("  3. Analyze key field uniqueness and overlap")
print("  4. Assess data leakage risks (see leakage_audit_report.csv)")
print("  5. Plan dataset integration strategy based on business requirements")
